# Compare Prediction with validation data


In [2]:
import pandas as pd
import geopandas as gpd

In [30]:
def summarize_points_in_bunn_types(points_gdf, polygons_gdf):
    total_points = len(points_gdf)
    print(f"total_points: {total_points}")

    for bunn_type in polygons_gdf["BunnType"].unique():
        sel_polygons = polygons_gdf[polygons_gdf["BunnType"] == bunn_type]

        points_in_polygons = gpd.sjoin(
            points_gdf,
            sel_polygons[["BunnType", "geometry"]],
            how="inner",
            predicate="within",
        )

        n_points = len(points_in_polygons)
        percent = (n_points / total_points) * 100 if total_points > 0 else 0

        print(f"Points in {bunn_type}: {n_points}")
        print(f"Percentage in {bunn_type}: {percent:.2f}% /n")

In [5]:
# Run once to setup points
pd.read_csv("./Stasjonsdata_bløtbunnsbasen 19.12.2025.csv").rename(
    columns={"y_coord_ny": "lat", "x_coord_ny": "lon"}
).drop_duplicates(subset=["lat", "lon"])[["lat", "lon", "DYP", "LOKALITET"]].to_csv(
    "./Stasjonsdata_blotbunnsbasen_points.csv", index=False
)

In [31]:
gdf_predict = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/nisjedata-substrat-xgbclassifier_norge_latest_25833.geo.parquet")

In [32]:
df = pd.read_csv("./Stasjonsdata_blotbunnsbasen_points.csv")
gdf_blotbunn = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
).to_crs(gdf_predict.crs)


In [33]:
summarize_points_in_bunn_types(gdf_blotbunn, gdf_predict)

total_points: 2400
Points in løsbunn: 2068
Percentage in løsbunn: 86.17% /n
Points in fastbunn: 148
Percentage in fastbunn: 6.17% /n
Points in blanding: 37
Percentage in blanding: 1.54% /n


In [34]:
gdf_aqua_hard = gpd.read_file("./aquamonitor_hardbunn_stations.geojson").to_crs(gdf_predict.crs)

summarize_points_in_bunn_types(gdf_aqua_hard, gdf_predict)


total_points: 285
Points in løsbunn: 69
Percentage in løsbunn: 24.21% /n
Points in fastbunn: 72
Percentage in fastbunn: 25.26% /n
Points in blanding: 3
Percentage in blanding: 1.05% /n


In [35]:
gdf_aqua_blot = gpd.read_file("./aquamonitor_blotbunn_stations.geojson").to_crs(gdf_predict.crs)

summarize_points_in_bunn_types(gdf_aqua_blot, gdf_predict)

total_points: 1506
Points in løsbunn: 1295
Percentage in løsbunn: 85.99% /n
Points in fastbunn: 95
Percentage in fastbunn: 6.31% /n
Points in blanding: 22
Percentage in blanding: 1.46% /n
